<a href="https://colab.research.google.com/github/scottn66/vgg11/blob/main/vgg11_from_scratch_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# VGG11 from scratch, v2

Same model and data as `vgg11_from_scratch_clean.ipynb` (VGG11 = config A of Simonyan & Zisserman 2014, plus BatchNorm, trained on a 20-class ImageNet subset), with the gaps against the paper closed and the training loop tightened.

**How the earlier notebooks compare to the paper**

| | Paper (config A) | `_clean` notebook | this notebook |
|---|---|---|---|
| Conv stack | 8 × conv3 (64-128-256-256-512-512-512-512), 5 max-pools | same ✅ | same ✅ (built from the paper's config table) |
| Classifier | FC 4096 → FC 4096 → FC 1000 | FC 4096 → 4096 → 20 ✅ | same ✅ |
| BatchNorm | no (paper predates BN) | after every conv | after every conv (`BATCH_NORM` flag) |
| Dropout | **0.5** after FC1 and FC2 | 0.0 (original notebook: none) | **0.5** (`DROPOUT`) |
| Weight init | zero-mean normal, biases 0 | PyTorch layer defaults | He init for convs, N(0, 0.01) for FCs (torchvision's VGG init, `INIT`) |
| Augmentation | random 224 crop of rescaled image, flip, RGB colour shift | RandomResizedCrop + flip | same as `_clean` |
| Optimizer | SGD m=0.9, wd 5e-4, batch 256, lr 0.01 ÷10 on plateau, 74 epochs | SGD m=0.9, wd 5e-4, batch 64, lr 0.01 cosine, 30 epochs | same as `_clean` |
| Reported accuracy | held-out test set | best epoch picked on val, **and reported on val** | best epoch picked on val, reported on **test** |

Parameter counts match torchvision's `vgg11` / `vgg11_bn` exactly (checked in section 3), so the layer stack itself was already right. What differed was dropout, initialization and evaluation. The init likely explains run0: with PyTorch's default init and no BatchNorm, activations shrink ~60× over the 8 conv layers (section 3c), so the logits start almost constant and the loss sits at ln 20 (run3 in the log tests this).

Also new: mixed precision + `channels_last` (faster on the T4), seeding, val loss / train accuracy tracking, history saved next to the checkpoint.

To reproduce run1 exactly, set `DROPOUT = 0.0`, `INIT = "default"`.

Contents: 1. Setup · 2. Data · 3. Model · 4. Sanity check · 5. Training · 6. Test-set evaluation · 7. Results log

## 1. Setup

In [ ]:
import os, json, time, random
import numpy as np
import torch, torch.nn as nn
import torchvision
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device, torch.__version__)

All settings in one place. Change **one** setting per run and record it in the results log (section 7).

In [ ]:
# ---- run config ----
RUN_NAME     = "run2_dropout"
EPOCHS       = 30
BATCH_SIZE   = 64
LR           = 0.01
WEIGHT_DECAY = 5e-4
DROPOUT      = 0.5         # paper: 0.5 after FC1 and FC2 (run1 used 0.0)
BATCH_NORM   = True        # not in the paper; False = original 2014 VGG11
INIT         = "kaiming"   # "kaiming" = torchvision VGG init, "default" = PyTorch layer defaults (run0/run1)
LABEL_SMOOTH = 0.0
AMP          = True        # mixed precision on GPU (big speedup on T4 tensor cores)
SEED         = 0
# --------------------

DATA      = "/content/data"
TRAIN_DIR = f"{DATA}/imagenet_train20a"
VAL_DIR   = f"{DATA}/imagenet_val20"
TEST_DIR  = f"{DATA}/imagenet_test20"
RUNS      = "/content/drive/MyDrive/vgg11/runs"

CONFIG = dict(RUN_NAME=RUN_NAME, EPOCHS=EPOCHS, BATCH_SIZE=BATCH_SIZE, LR=LR, WEIGHT_DECAY=WEIGHT_DECAY,
              DROPOUT=DROPOUT, BATCH_NORM=BATCH_NORM, INIT=INIT, LABEL_SMOOTH=LABEL_SMOOTH, AMP=AMP, SEED=SEED)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)   # torch.manual_seed also seeds CUDA
torch.backends.cudnn.benchmark = True   # fixed 224x224 inputs: let cuDNN pick the fastest conv kernels

## 2. Data
Copy the zips from Drive and unzip them to local disk (much faster reads than Drive).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p {DATA} {RUNS}
!cp /content/drive/MyDrive/vgg11/data/imagenet_*20.zip {DATA}/
%cd {DATA}
!for z in imagenet_*20.zip; do unzip -q -o "$z"; done
!ls; ls imagenet_train20a | wc -l; ls imagenet_val20 | wc -l; ls imagenet_test20 | head -3

Train is in `ImageFolder` layout (`<wnid>/img.JPEG`). Val is one flat folder, so the label comes from the filename prefix. `labeled_folder` handles either layout, so the test set works whichever way it was packed.

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]   # ImageNet stats

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


class FlatFolder(Dataset):
    """Flat image folder; label = wnid prefix of the filename, e.g. n01737021_10767.JPEG."""

    def __init__(self, root, class_to_idx, tf):
        self.root, self.c2i, self.tf = root, class_to_idx, tf
        self.files = sorted(f for f in os.listdir(root) if not f.startswith("."))
        unknown = {f.split("_")[0] for f in self.files} - class_to_idx.keys()
        if unknown:
            raise ValueError(f"{root}: filename prefixes not among the train classes: {sorted(unknown)[:5]}")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        name = self.files[i]
        img = Image.open(os.path.join(self.root, name)).convert("RGB")
        return self.tf(img), self.c2i[name.split("_")[0]]


def labeled_folder(root, class_to_idx, tf):
    """ImageFolder layout (<wnid>/img.JPEG) or a flat folder with wnid-prefixed filenames."""
    if any(os.path.isdir(os.path.join(root, d)) for d in os.listdir(root)):
        ds = datasets.ImageFolder(root, transform=tf)
        assert ds.class_to_idx == class_to_idx, f"{root}: class folders differ from train"
        return ds
    return FlatFolder(root, class_to_idx, tf)


train_ds = datasets.ImageFolder(TRAIN_DIR, transform=train_tf)
val_ds   = labeled_folder(VAL_DIR, train_ds.class_to_idx, eval_tf)
test_ds  = labeled_folder(TEST_DIR, train_ds.class_to_idx, eval_tf) if os.path.isdir(TEST_DIR) else None

loader_kw = dict(batch_size=BATCH_SIZE, num_workers=os.cpu_count(), pin_memory=(device == "cuda"),
                 persistent_workers=True)
train_dl = DataLoader(train_ds, shuffle=True, generator=torch.Generator().manual_seed(SEED), **loader_kw)
val_dl   = DataLoader(val_ds, shuffle=False, **loader_kw)
test_dl  = DataLoader(test_ds, shuffle=False, **loader_kw) if test_ds is not None else None

# check
print(len(train_ds), len(val_ds), len(test_ds) if test_ds else "no test set", len(train_ds.classes))  # expect 6000 1000 ... 20
x, y = train_ds[0]; print(x.shape, y)                                                                  # expect (3, 224, 224) 0

## 3. Model
### 3a. Layers from the paper's config table
Each number is a 3×3 conv (stride 1, padding 1) with that many output channels; `"M"` is a 2×2 max-pool with stride 2. Config A is VGG11; the deeper ones come for free.

In [ ]:
CFGS = {
    "A": [64, "M", 128, "M", 256, 256, "M", 512, 512, "M", 512, 512, "M"],                                          # VGG11
    "B": [64, 64, "M", 128, 128, "M", 256, 256, "M", 512, 512, "M", 512, 512, "M"],                                 # VGG13
    "D": [64, 64, "M", 128, 128, "M", 256, 256, 256, "M", 512, 512, 512, "M", 512, 512, 512, "M"],                  # VGG16
    "E": [64, 64, "M", 128, 128, "M", 256, 256, 256, 256, "M", 512, 512, 512, 512, "M", 512, 512, 512, 512, "M"],   # VGG19
}


def conv_block(c_in, c_out, batch_norm=True):
    layers = [nn.Conv2d(c_in, c_out, kernel_size=3, padding=1)]
    if batch_norm:
        layers.append(nn.BatchNorm2d(c_out))
    layers.append(nn.ReLU(inplace=True))
    return nn.Sequential(*layers)


def make_features(cfg, batch_norm=True):
    layers, c_in = [], 3
    for v in cfg:
        if v == "M":
            layers.append(nn.MaxPool2d(2, 2))
        else:
            layers.append(conv_block(c_in, v, batch_norm))
            c_in = v
    return nn.Sequential(*layers)


def make_classifier(num_classes=20, dropout=0.5):
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(512 * 7 * 7, 4096), nn.ReLU(inplace=True), nn.Dropout(dropout),
        nn.Linear(4096, 4096),        nn.ReLU(inplace=True), nn.Dropout(dropout),
        nn.Linear(4096, num_classes),   # no softmax: CrossEntropyLoss applies log-softmax itself
    )


def init_weights(model):
    """torchvision's VGG init: He init for convs keeps the activation scale through ReLUs; N(0, 0.01) for FCs."""
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            nn.init.zeros_(m.bias)
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0, 0.01); nn.init.zeros_(m.bias)


class VGG(nn.Module):
    def __init__(self, cfg, num_classes=20, batch_norm=True, dropout=0.5, init="kaiming"):
        super().__init__()
        self.features = make_features(cfg, batch_norm)
        self.avgpool = nn.AdaptiveAvgPool2d((7, 7))   # no-op at 224x224; lets other input sizes reach the classifier
        self.classifier = make_classifier(num_classes, dropout)
        if init == "kaiming":
            init_weights(self)

    def forward(self, x):
        return self.classifier(self.avgpool(self.features(x)))


def VGG11(**kw):
    return VGG(CFGS["A"], **kw)

### 3b. Check against the paper and torchvision
Same parameter shapes, in the same order, as torchvision's reference implementation. State-dict keys also match the `_clean` notebook, so its checkpoints load into this model.

In [ ]:
n_params = lambda m: sum(p.numel() for p in m.parameters())
shapes = lambda m: [tuple(p.shape) for p in m.parameters()]

m = VGG11()
print(m(torch.randn(2, 3, 224, 224)).shape)                                  # expect (2, 20)
print(f"{n_params(m):,}")                                                    # expect 128,853,780

assert n_params(VGG11(batch_norm=False, num_classes=1000)) == 132_863_336    # paper Table 2: "133M"
assert n_params(VGG11(batch_norm=False)) == 128_848_276                      # 20 classes, no BN
assert shapes(VGG11(batch_norm=False)) == shapes(torchvision.models.vgg11(num_classes=20))
assert shapes(VGG11(batch_norm=True)) == shapes(torchvision.models.vgg11_bn(num_classes=20))
print("matches torchvision vgg11 / vgg11_bn")

### 3c. Why run0 (no BatchNorm) was stuck at chance
Push a random batch through VGG11 *without* BatchNorm and print the std of each conv layer's output. With PyTorch's default init every layer shrinks the signal, so by the classifier the logits are almost identical for every image: loss = ln 20 and tiny gradients. He init keeps the scale roughly constant. (BatchNorm re-normalizes after every conv, which is why run1 trained either way.)

In [ ]:
@torch.no_grad()
def conv_output_stds(model, x):
    stds = []
    for block in model.features:
        x = block(x)
        if isinstance(block, nn.Sequential):   # a conv block (pools are bare MaxPool2d)
            stds.append(x.std().item())
    return stds, model.classifier(model.avgpool(x)).std().item()

x = torch.randn(4, 3, 224, 224)
for init in ["default", "kaiming"]:
    stds, logit_std = conv_output_stds(VGG11(batch_norm=False, init=init).eval(), x)
    print(f"{init:8s} conv stds: {' '.join(f'{s:.3f}' for s in stds)}   logit std: {logit_std:.4f}")

## 4. Sanity check: overfit one batch
If the model can't drive the loss near 0 on 8 images, something is broken. Dropout is off here so this tests only the model and the update step.

In [ ]:
model = VGG11(batch_norm=BATCH_NORM, dropout=0.0, init=INIT).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.005, momentum=0.9)

x, y = next(iter(train_dl))
x, y = x[:8].to(device), y[:8].to(device)

model.train()
for step in range(300):
    loss = criterion(model(x), y)
    optimizer.zero_grad()   # clear old gradients
    loss.backward()         # compute new gradients
    optimizer.step()        # update weights
    if step % 20 == 0:
        print(step, round(loss.item(), 4))

## 5. Training
SGD + momentum, weight decay, cosine learning-rate schedule, mixed precision. Saves the best checkpoint by **val** accuracy; the number to report comes from the **test** set in section 6.

Train accuracy is measured on augmented images with dropout on, so it reads lower than the model's real fit to the training set.

If the loss spikes in the first epochs with dropout on, try `LR = 0.005`.

In [ ]:
use_amp = AMP and device == "cuda"

model = VGG11(batch_norm=BATCH_NORM, dropout=DROPOUT, init=INIT).to(device, memory_format=torch.channels_last)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
optimizer = torch.optim.SGD(model.parameters(), lr=LR, momentum=0.9, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler(device, enabled=use_amp)   # rescales the fp16 loss so small gradients don't underflow


def to_device(x, y):
    return x.to(device, non_blocking=True, memory_format=torch.channels_last), y.to(device, non_blocking=True)


def train_one_epoch(model, loader):
    model.train()
    loss_sum = correct = total = 0
    for x, y in loader:
        x, y = to_device(x, y)
        with torch.autocast(device, enabled=use_amp):
            logits = model(x)
            loss = criterion(logits, y)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        loss_sum += loss.item() * y.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        total += y.size(0)
    return loss_sum / total, correct / total


@torch.no_grad()
def evaluate(model, loader):
    model.eval()                          # BatchNorm uses running stats, dropout off
    loss_sum = correct = total = 0
    for x, y in loader:
        x, y = to_device(x, y)
        with torch.autocast(device, enabled=use_amp):
            logits = model(x)
            loss = criterion(logits, y)
        loss_sum += loss.item() * y.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        total += y.size(0)
    return loss_sum / total, correct / total


ckpt_path = f"{RUNS}/{RUN_NAME}_best.pt"
history, best = [], 0.0
for epoch in range(EPOCHS):
    t0 = time.time()
    lr = scheduler.get_last_lr()[0]
    train_loss, train_acc = train_one_epoch(model, train_dl)
    val_loss, val_acc = evaluate(model, val_dl)
    scheduler.step()                      # once per epoch

    history.append(dict(epoch=epoch + 1, lr=lr, train_loss=train_loss, train_acc=train_acc,
                        val_loss=val_loss, val_acc=val_acc))
    if val_acc > best:
        best = val_acc
        torch.save({"model": model.state_dict(), "epoch": epoch + 1, "val_acc": val_acc, "config": CONFIG}, ckpt_path)
    print(f"epoch {epoch+1:2d}  lr {lr:.4f}  train loss {train_loss:.3f} acc {train_acc:.3f}  "
          f"val loss {val_loss:.3f} acc {val_acc:.3f}  ({time.time()-t0:.0f}s)")

with open(f"{RUNS}/{RUN_NAME}_history.json", "w") as f:
    json.dump({"config": CONFIG, "history": history}, f, indent=1)
print(f"best val acc: {best:.3f}")

In [ ]:
ep = [h["epoch"] for h in history]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for split in ["train", "val"]:
    ax[0].plot(ep, [h[f"{split}_loss"] for h in history], label=split)
    ax[1].plot(ep, [h[f"{split}_acc"] for h in history], label=split)
ax[0].set_title("loss"); ax[1].set_title("accuracy")
for a in ax:
    a.set_xlabel("epoch"); a.legend()
plt.tight_layout(); plt.show()

## 6. Test-set evaluation
The val set chose the checkpoint, so best-val accuracy is biased upward (the max over 30 noisy evaluations). The test set was never looked at, so its accuracy is the honest number.

In [ ]:
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt["model"])
print(f"best checkpoint: epoch {ckpt['epoch']}, val acc {ckpt['val_acc']:.3f}")

if test_dl is not None:
    test_loss, test_acc = evaluate(model, test_dl)
    print(f"test acc {test_acc:.3f}  (loss {test_loss:.3f}, {len(test_ds)} images)")
else:
    print(f"no test set at {TEST_DIR}")

## 7. Results log

| Run | Change | Epochs | Best val acc | Test acc |
|---|---|---|---|---|
| run0 | no BatchNorm, default init | 30 | 0.050 (stuck at chance, loss = ln 20) | |
| run1_bn | + BatchNorm | 30 | **0.759** (original notebook) / 0.761 (`_clean` rerun) | |
| run2_dropout | + dropout 0.5, He init | 30 | | |
| run3 | no BatchNorm, He init (does init alone fix run0?) | 30 | | |
| run4 | 50 epochs | 50 | | |